# Workers Analysis

Analyse worker statistics from MCR experiment checkpoints.

Each checkpoint directory contains `redis_backup_db10.json` (LOGS) with per-iteration
worker events.  This notebook loads that data and produces summary tables and plots.

In [ ]:
from pathlib import Path
from etl.loader import etl_from_dir, list_checkpoints

RESULTS_DIR = Path("results")

# List available checkpoints and pick one
checkpoints = list_checkpoints(RESULTS_DIR)
print(f"Found {len(checkpoints)} checkpoints:")
for i, cp in enumerate(checkpoints):
    print(f"  [{i}] {cp.relative_to(RESULTS_DIR)}")

# Select checkpoint (edit index or set CHECKPOINT_DIR directly)
CHECKPOINT_DIR = checkpoints[0] if checkpoints else None
print(f"\nUsing: {CHECKPOINT_DIR}")

In [ ]:
# Load checkpoint into structured db dict
db = etl_from_dir(CHECKPOINT_DIR, verbose=True)

In [ ]:
from etl.logs_loader import build_db10_worker_report
from etl.constants import DB_NAMES
import json
from pathlib import Path

# Build worker report from DB10 entries stored in the db dict
# The logs_loader expects the old ZIP-based signature; we adapt here by
# reconstructing the minimal structures it needs from the directory.

checkpoint_dir = Path(db['_checkpoint_dir'])
db10_path = checkpoint_dir / 'redis_backup_db10.json'
manifest_path = checkpoint_dir / 'manifest.json'

manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
db10_data = json.loads(db10_path.read_text()) if db10_path.exists() else {'entries': []}

# files_map for compatibility: map '10' -> filename
files_map = manifest.get('files', {})
if '10' not in files_map:
    files_map['10'] = 'redis_backup_db10.json'

mock_manifest = {'files': files_map}
mock_backups = {'redis_backup_db10.json': db10_data}
mock_archive_data = {'zip_path': str(checkpoint_dir)}
mock_prefix = ''

report = build_db10_worker_report(
    selected_zip_name=checkpoint_dir.name,
    selected_manifest=mock_manifest,
    selected_backups=mock_backups,
    selected_archive_data=mock_archive_data,
    selected_manifest_prefix=mock_prefix,
    max_events=None
)

# Store report in db for render_worker_report compatibility
db[DB_NAMES.get(10)] = report
print(f"Worker report built: {report.get('worker_count', 0)} workers")

In [ ]:
from etl.logs_loader import render_worker_report

workers_table, workers_table2, workers_table3, plots = render_worker_report(db)
display(workers_table)
display(workers_table2)
display(workers_table3)